# Notebook 01 - Data Collection

**Research Question:** Which factors influence rental prices in Switzerland?

This notebook covers:
- **Requirement 1:** Collection of real-world data (web scraper)
- **Requirement 2:** Data preparation with regular expressions
- **Requirement 3:** Python built-in data structures (list, dict, set, tuple)
- **Requirement 4:** Conditional statements, loops and loop control
- **Requirement 5:** OOP - classes Listing, ImmobilienScraper, ImmobilienDB
- **Bonus B1:** Web scraper (Flatfox.ch)
- **Bonus B2:** SQLite database with SQL queries


In [ ]:
import sys
sys.path.insert(0, '..')  # make src module importable

import pandas as pd
import re
from pathlib import Path

from src.scraper import ImmobilienScraper
from src.database import ImmobilienDB
from src.models import Inserat

print('Imports OK')

## 1. Data Collection via Web Scraper

The scraper attempts to fetch real listings from Flatfox.ch.
If the connection fails, it automatically falls back to realistic sample data
based on Swiss Federal Statistical Office (FSO) price statistics 2023/24.

In [ ]:
scraper = ImmobilienScraper(delay_sek=1.5)
cities = ['zuerich', 'bern', 'basel', 'genf', 'luzern', 'lausanne', 'winterthur', 'lugano']
df_raw = scraper.scrape_alle_staedte(staedte=cities, n_beispiel=50)
print(f'Dataset: {df_raw.shape[0]} rows x {df_raw.shape[1]} columns')
df_raw.head()

## 2. Regex Parsing Demo

**Requirement 2:** Shows explicitly how regular expressions convert raw strings to numeric values.
All four parsers are implemented in `src/models.py` inside the `Inserat` class.

In [ ]:
raw_strings = [
    ("CHF 2'450.- / month", 'price'),
    ('1 800 CHF/mo.',        'price'),
    ('85 m2',                'area'),
    ('120m2',                'area'),
    ('3.5 rooms',            'rooms'),
    ('4-room',               'rooms'),
    ('8001 Zurich, ZH',      'location'),
    ('3011 Bern',            'location'),
]
patterns = {
    'price':    (r"['\s]", r"(\d+(?:\.\d+)?)"),
    'area':     (None,     r"(\d+(?:[.,]\d+)?)\s*m[2]?"),
    'rooms':    (None,     r"(\d+(?:[.,]\d+)?)"),
    'location': (None,     r"(\d{4})\s+([^\,]+)"),
}
print('Regex Parsing Demo:')
print('-' * 50)
for raw_str, typ in raw_strings:
    clean_pat, search_pat = patterns[typ]
    s = re.sub(clean_pat, '', raw_str) if clean_pat else raw_str
    m = re.search(search_pat, s)
    result = m.group(1) if m else 'no match'
    print(f'  {raw_str:30} -> {result}')

## 3. Python Built-in Data Structures

**Requirement 3:** Explicit demonstration of list, set, dict, and tuple.

In [ ]:
# LIST - ordered collection of all city entries
cities_list = list(df_raw['stadt'].dropna())
print(f'List (first 5): {cities_list[:5]}')

# SET - unique cities without duplicates
unique_cities = set(cities_list)
print(f'\nSet (unique cities): {sorted(unique_cities)}')

# DICT - average price per city, sorted descending
prices_dict = {
    city: round(df_raw[df_raw['stadt'] == city]['preis_chf'].mean(), 0)
    for city in unique_cities
    if len(df_raw[df_raw['stadt'] == city]) > 0
}
prices_dict = dict(sorted(prices_dict.items(), key=lambda x: x[1], reverse=True))
print('\nDict (avg price per city):')
for city, price in prices_dict.items():
    print(f'  {city:15} CHF {price:,.0f}')

# TUPLE - immutable price range (min, max)
price_range = (df_raw['preis_chf'].min(), df_raw['preis_chf'].max())
print(f'\nTuple (price range): CHF {price_range[0]:,.0f} - CHF {price_range[1]:,.0f}')

## 4. Database - Store Listings & SQL Queries

**Bonus B2:** SQLite database with SQL queries (CREATE, INSERT OR IGNORE, GROUP BY, LEFT JOIN).

In [ ]:
db = ImmobilienDB()
db.kantone_befuellen()

listings = []
for _, row in df_raw.iterrows():
    ins = Inserat(
        titel=str(row.get('titel', '')),
        preis_raw=str(row.get('preis_chf', '')),
        flaeche_raw=str(row.get('flaeche_m2', '')),
        zimmer_raw=str(row.get('zimmer_anzahl', '')),
        ort_raw=f"{row.get('plz', '')} {row.get('stadt', '')}",
        url=str(row.get('url', '')),
    )
    listings.append(ins)

db.bulk_speichern(listings)
print(f'{db}')

In [ ]:
# SQL GROUP BY - price statistics per city (AVG, MIN, MAX, COUNT)
print('SQL query: SELECT city, AVG, MIN, MAX, COUNT GROUP BY city')
db.preisstatistik_pro_stadt().head(10)

## 5. Save Dataset for Next Notebooks

In [ ]:
df_final = db.alle_laden()
Path('../data').mkdir(exist_ok=True)
df_final.to_csv('../data/inserate_roh.csv', index=False)
print(f'Saved: data/inserate_roh.csv ({len(df_final)} rows)')
print(f'Columns: {list(df_final.columns)}')
df_final.describe().round(1)